In [5]:
# ################################################################################
# #  MUMBAI HEAT STRESS × AIR POLLUTION — CORRELATION ANALYSIS                 #
# #  Stations : 43003 Santacruz  (IDW from MH016 + MH018)                      #
# #             43057 Colaba     (direct: MH013)                                #
# #  Targets  : HI (Heat Index)  +  PET (Physiological Equivalent Temperature) #
# #  Period   : June 2019 – Dec 2025 (MPCB overlap)                            #
# ################################################################################



# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install & Imports                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# !pip install scipy seaborn matplotlib pandas numpy scikit-learn --quiet

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr
from sklearn.decomposition import PCA
from collections import Counter
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 150, 'font.size': 10,
                     'axes.titlesize': 11, 'axes.labelsize': 10})

print('✅ Imports ready.')



# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Configuration                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

BASE_DIR    = '/content/drive/MyDrive/Major_project_imd'
INPUT_DIR   = os.path.join(BASE_DIR, 'Class_balancing_output')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'pollution_correlation_analysis')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── IMD files ──────────────────────────────────────────────────────────────────
IMD_43003 = os.path.join(INPUT_DIR, '43003_with_HI_PET_categories.csv')
IMD_43057 = os.path.join(INPUT_DIR, '43057_with_HI_PET_categories.csv')

# ── MPCB files ─────────────────────────────────────────────────────────────────
# Santacruz IDW: MH016 (Vile Parle West, 2.5 km) + MH018 (Airport T2, 1.5 km)
MH016_FILE  = '/content/drive/MyDrive/Major_project_imd/MPCB/MH016.csv'
MH018_FILE  = '/content/drive/MyDrive/Major_project_imd/Major_project_imd/MPCB/MH018.csv'
# Colaba: MH013 (direct)
MH013_FILE  = '/content/drive/MyDrive/Major_project_imd/MPCB/MH013.csv'

# IDW distances for Santacruz
D_MH016, D_MH018 = 2.5, 1.5     # km
W_MH016 = (1/D_MH016**2) / (1/D_MH016**2 + 1/D_MH018**2)   # 0.265
W_MH018 = (1/D_MH018**2) / (1/D_MH016**2 + 1/D_MH018**2)   # 0.735

# Synoptic hour mapping  HR-code → IST hour
HR_IST_MAP = {0:5, 12:8, 24:11, 36:14, 48:17, 60:20, 72:23, 84:2}

# Pollutant columns
POLL_COLS   = ['PM2.5','PM10','NO2','NH3','SO2','CO','Ozone']
MET_COLS    = ['AT','RH','WS','WD']
ALL_MPCB    = POLL_COLS + MET_COLS
TARGETS     = ['HI', 'PET']

# NAQI breakpoints (CPCB 2024)
CPCB_BP = {
    'PM2.5': [(0,30,0,50),(30,60,51,100),(60,90,101,200),
              (90,120,201,300),(120,250,301,400),(250,500,401,500)],
    'PM10':  [(0,50,0,50),(50,100,51,100),(100,250,101,200),
              (250,350,201,300),(350,430,301,400),(430,600,401,500)],
    'NO2':   [(0,40,0,50),(40,80,51,100),(80,180,101,200),
              (180,280,201,300),(280,400,301,400),(400,800,401,500)],
    'SO2':   [(0,40,0,50),(40,80,51,100),(80,380,101,200),
              (380,800,201,300),(800,1600,301,400),(1600,2100,401,500)],
    'CO':    [(0,1.0,0,50),(1.0,2.0,51,100),(2.0,10.0,101,200),
              (10.0,17.0,201,300),(17.0,34.0,301,400),(34.0,50.0,401,500)],
    'Ozone': [(0,50,0,50),(50,100,51,100),(100,168,101,200),
              (168,208,201,300),(208,748,301,400),(748,1000,401,500)],
    'NH3':   [(0,200,0,50),(200,400,51,100),(400,800,101,200),
              (800,1200,201,300),(1200,1800,301,400),(1800,2400,401,500)],
}

# HI / PET category labels
HI_CATS  = {0:'Low Risk', 1:'Moderate', 2:'High', 3:'Very High'}
PET_CATS = {0:'Comfortable', 1:'Slightly Warm', 2:'Warm', 3:'Hot', 4:'Very Hot'}

print(f'IDW weights: MH016={W_MH016:.3f}  MH018={W_MH018:.3f}')
print(f'Output: {OUTPUT_DIR}')
print('✅ Configuration ready.')



# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Helper Functions                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def naqi(x, bps):
    if pd.isna(x): return np.nan
    for Cl,Ch,Il,Ih in bps:
        if Cl <= x <= Ch:
            return ((Ih-Il)/(Ch-Cl))*(x-Cl)+Il
    return 500.0 if x > bps[-1][1] else np.nan

def compute_aqi(df_poll):
    """Compute NAQI sub-indices and AQI_max."""
    for p, bps in CPCB_BP.items():
        if p in df_poll.columns:
            df_poll[f'NAQI_{p}'] = df_poll[p].apply(lambda x: naqi(x, bps))
    naqi_cols = [f'NAQI_{p}' for p in CPCB_BP if f'NAQI_{p}' in df_poll.columns]
    df_poll['AQI_max'] = df_poll[naqi_cols].max(axis=1)
    return df_poll

def load_mpcb_station(filepath, code):
    """Load, clean and index a single MPCB station CSV."""
    d = pd.read_csv(filepath)
    d.columns = d.columns.str.strip()
    d['DATETIME'] = pd.to_datetime(d['Date'], format='%m/%d/%Y %H:%M', errors='coerce')
    nat = d['DATETIME'].isna()
    if nat.any():
        d.loc[nat,'DATETIME'] = pd.to_datetime(
            d.loc[nat,'Date'], infer_datetime_format=True, errors='coerce')
    for c in ALL_MPCB:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors='coerce')
    for c in POLL_COLS:
        if c in d.columns: d.loc[d[c]<0, c] = np.nan
    if 'AT' in d.columns: d.loc[(d['AT']<10)|(d['AT']>50),'AT'] = np.nan
    if 'RH' in d.columns: d.loc[(d['RH']<0)|(d['RH']>100),'RH'] = np.nan
    d = (d.sort_values('DATETIME')
          .drop_duplicates('DATETIME')
          .set_index('DATETIME'))
    avail = [c for c in ALL_MPCB if c in d.columns]
    print(f'  {code}: {len(d):,} rows  {d.index.min().date()} → {d.index.max().date()}')
    return d[avail]

def idw_interpolate(df_a, df_b, wa, wb):
    """Inverse-distance-weighted merge of two MPCB DataFrames."""
    idx = df_a.index.union(df_b.index)
    a   = df_a.reindex(idx).ffill(limit=2).bfill(limit=2)
    b   = df_b.reindex(idx).ffill(limit=2).bfill(limit=2)
    common = [c for c in ALL_MPCB if c in a.columns and c in b.columns]
    out = pd.DataFrame(index=idx)
    for c in common:
        out[c] = wa*a[c] + wb*b[c]
    return out

def build_imd_datetime(df_imd):
    """Reconstruct IST datetime from YEAR/MN/DT/HR columns."""
    hr_ist = df_imd['HR'].map(HR_IST_MAP).fillna(0).astype(int)
    dt = (pd.to_datetime(df_imd[['YEAR','MN','DT']]
                         .rename(columns={'YEAR':'year','MN':'month','DT':'day'}),
                         errors='coerce')
          + pd.to_timedelta(hr_ist, unit='h')
          + pd.Timedelta(minutes=30))
    return dt.dt.round('H')

def compute_hapsi(df_m, alpha, beta, gamma):
    """HAPSI = α·HI_norm + β·PET_norm + γ·AQI_norm (0–100 scale)."""
    def norm(s, lo, hi): return (s.clip(lo,hi)-lo)/(hi-lo)
    hi_n  = norm(df_m['HI'],      20,  55)
    pet_n = norm(df_m['PET'],     15,  65)
    aqi_n = norm(df_m['AQI_max'],  0, 500)
    return (alpha*hi_n + beta*pet_n + gamma*aqi_n)*100

def pca_weights(df_m):
    """Compute PCA-based weights for HAPSI."""
    mask = df_m[['HI','PET','AQI_max']].notna().all(axis=1)
    if mask.sum() < 200: return 0.35, 0.35, 0.30
    def norm(s,lo,hi): return (s.clip(lo,hi)-lo)/(hi-lo)
    X = np.column_stack([norm(df_m.loc[mask,'HI'],20,55),
                         norm(df_m.loc[mask,'PET'],15,65),
                         norm(df_m.loc[mask,'AQI_max'],0,500)])
    w = np.abs(PCA(n_components=3).fit(X).components_[0])
    w /= w.sum()
    return float(w[0]), float(w[1]), float(w[2])

print('✅ Helper functions ready.')





✅ Imports ready.
IDW weights: MH016=0.265  MH018=0.735
Output: /content/drive/MyDrive/Major_project_imd/pollution_correlation_analysis
✅ Configuration ready.
✅ Helper functions ready.


In [6]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Load & Prepare Data for Both Stations                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

STATION_DATA = {}   # dict keyed by station_id

for STATION_ID, STATION_NAME, IMD_FILE, mpcb_src in [
    ('43003', 'Santacruz', IMD_43003, 'IDW'),
    ('43057', 'Colaba',    IMD_43057, 'direct'),
]:
    print(f'\n{"="*60}')
    print(f'  Loading {STATION_NAME} ({STATION_ID})')
    print(f'{"="*60}')

    # ── Load IMD ──────────────────────────────────────────────────────────────
    df_imd = pd.read_csv(IMD_FILE)
    df_imd['_dt'] = build_imd_datetime(df_imd)
    df_imd = df_imd.dropna(subset=['_dt'])
    df_imd = df_imd[~df_imd['_dt'].duplicated(keep='first')]
    print(f'  IMD rows: {len(df_imd):,}  ({df_imd["YEAR"].min()}–{df_imd["YEAR"].max()})')

    # ── Load & IDW / direct MPCB ──────────────────────────────────────────────
    if STATION_ID == '43003':
        print('  Loading MH016 + MH018 for IDW ...')
        mh016 = load_mpcb_station(MH016_FILE, 'MH016')
        mh018 = load_mpcb_station(MH018_FILE, 'MH018')
        df_poll = idw_interpolate(mh016, mh018, W_MH016, W_MH018)
        print(f'  IDW done: {len(df_poll):,} hourly rows')
    else:
        print('  Loading MH013 (Colaba direct) ...')
        df_poll = load_mpcb_station(MH013_FILE, 'MH013')

    # ── Compute NAQI + AQI_max ────────────────────────────────────────────────
    df_poll = compute_aqi(df_poll.copy())
    df_poll.index = df_poll.index.round('H')
    df_poll = df_poll[~df_poll.index.duplicated(keep='first')]

    # ── LEFT JOIN MPCB onto IMD ───────────────────────────────────────────────
    merge_cols = POLL_COLS + ['AQI_max'] + [f'NAQI_{p}' for p in CPCB_BP]
    merge_cols = [c for c in merge_cols if c in df_poll.columns]
    poll_reset = df_poll[merge_cols].reset_index().rename(columns={'DATETIME':'_dt'})
    poll_reset['_dt'] = pd.to_datetime(poll_reset['_dt']).dt.round('H')
    poll_reset = poll_reset.drop_duplicates('_dt')
    df_merge = df_imd.merge(poll_reset, on='_dt', how='left')

    # ── Compute HAPSI ─────────────────────────────────────────────────────────
    mask = df_merge[['HI','PET','AQI_max']].notna().all(axis=1)
    alpha, beta, gamma = pca_weights(df_merge)
    print(f'  PCA weights: α(HI)={alpha:.3f}  β(PET)={beta:.3f}  γ(AQI)={gamma:.3f}')
    df_merge['HAPSI'] = np.nan
    df_merge.loc[mask,'HAPSI'] = compute_hapsi(
        df_merge.loc[mask], alpha, beta, gamma).values

    # ── Pollution-period only (where all poll cols available) ─────────────────
    all_poll = POLL_COLS + ['AQI_max','HAPSI']
    poll_period = df_merge[df_merge[POLL_COLS].notna().all(axis=1)].copy()
    poll_period['Month'] = poll_period['MN']
    poll_period['Season'] = poll_period['MN'].map(
        {12:'Winter',1:'Winter',2:'Winter',
         3:'Pre-monsoon',4:'Pre-monsoon',5:'Pre-monsoon',
         6:'Monsoon',7:'Monsoon',8:'Monsoon',9:'Monsoon',
         10:'Post-monsoon',11:'Post-monsoon'})

    print(f'  Pollution-period rows: {len(poll_period):,}  '
          f'({poll_period["YEAR"].min()}–{poll_period["YEAR"].max()})')
    print(f'  HI match: {poll_period["HI"].notna().sum():,}  '
          f'PET match: {poll_period["PET"].notna().sum():,}')

    STATION_DATA[STATION_ID] = {
        'name': STATION_NAME,
        'df':   poll_period,
        'alpha': alpha, 'beta': beta, 'gamma': gamma,
    }

print('\n✅ Both stations loaded and merged.')


  Loading Santacruz (43003)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Major_project_imd/Class_balancing_output/43003_with_HI_PET_categories.csv'

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4b — Synoptic Period Alignment                                       ║
# ║                                                                            ║
# ║  PROBLEM:  Santacruz (43003) has 8 synoptic periods/day → ~155k rows      ║
# ║            Colaba   (43057) has only 3 synoptic periods/day → ~55k rows   ║
# ║                                                                            ║
# ║  FIX:      For ALL station comparisons, filter Santacruz to the same      ║
# ║            3 synoptic HR codes that Colaba observes, so both datasets     ║
# ║            represent identical observation windows before comparing.       ║
# ║            Original full Santacruz data saved as df_original.             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print('\n' + '='*60)
print('  SYNOPTIC PERIOD ALIGNMENT — as per Jaya Mam correction')
print('='*60)

# ── Step 1: Detect each station's synoptic HR codes ──────────────────────────
colaba_hrs    = set(STATION_DATA['43057']['df']['HR'].dropna().astype(int).unique())
santa_hrs_all = set(STATION_DATA['43003']['df']['HR'].dropna().astype(int).unique())

print(f'\n  Colaba    synoptic HR codes ({len(colaba_hrs)} periods) : '
      f'{sorted(colaba_hrs)}'
      f'  → IST {sorted([HR_IST_MAP.get(h, h) for h in colaba_hrs])} h')
print(f'  Santacruz synoptic HR codes ({len(santa_hrs_all)} periods) : '
      f'{sorted(santa_hrs_all)}'
      f'  → IST {sorted([HR_IST_MAP.get(h, h) for h in santa_hrs_all])} h')

# ── Step 2: Find the common (overlapping) HR codes ───────────────────────────
common_hrs = colaba_hrs & santa_hrs_all
print(f'\n  Common HR codes ({len(common_hrs)} periods) : '
      f'{sorted(common_hrs)}'
      f'  → IST {sorted([HR_IST_MAP.get(h, h) for h in common_hrs])} h')

# ── Step 3: Preserve original Santacruz (all 8 periods) ─────────────────────
STATION_DATA['43003']['df_original'] = STATION_DATA['43003']['df'].copy()

# ── Step 4: Filter Santacruz to only common synoptic periods ─────────────────
santa_filtered = STATION_DATA['43003']['df'][
    STATION_DATA['43003']['df']['HR'].isin(common_hrs)
].copy()

print(f'\n  Row counts after alignment:')
print(f'    Santacruz — original  (all {len(santa_hrs_all)} periods) : '
      f'{len(STATION_DATA["43003"]["df"]):>8,} rows')
print(f'    Santacruz — aligned   ({len(common_hrs)} periods)         : '
      f'{len(santa_filtered):>8,} rows')
print(f'    Colaba    — unchanged ({len(colaba_hrs)} periods)          : '
      f'{len(STATION_DATA["43057"]["df"]):>8,} rows')

# ── Step 5: Recompute PCA-based HAPSI weights on filtered subset ─────────────
alpha_new, beta_new, gamma_new = pca_weights(santa_filtered)
print(f'\n  Recomputed PCA weights for aligned Santacruz:')
print(f'    α(HI) = {alpha_new:.3f}   β(PET) = {beta_new:.3f}   γ(AQI) = {gamma_new:.3f}')

mask_f = santa_filtered[['HI', 'PET', 'AQI_max']].notna().all(axis=1)
santa_filtered['HAPSI'] = np.nan
santa_filtered.loc[mask_f, 'HAPSI'] = compute_hapsi(
    santa_filtered.loc[mask_f], alpha_new, beta_new, gamma_new
).values

# ── Step 6: Update STATION_DATA — all downstream cells use aligned df ────────
STATION_DATA['43003']['df']    = santa_filtered
STATION_DATA['43003']['alpha'] = alpha_new
STATION_DATA['43003']['beta']  = beta_new
STATION_DATA['43003']['gamma'] = gamma_new

print(f'\n  ✅ Santacruz now filtered to {len(common_hrs)} synoptic periods matching Colaba.')
print('     All plots (PLOT 1–12) and correlation tables will use this aligned dataset.')
print('     To access full 8-period data: STATION_DATA["43003"]["df_original"]')
print('='*60)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Spearman Correlation Tables                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

POLL_FEATURES = POLL_COLS + ['AQI_max','HAPSI']
CORR_RESULTS  = {}

print('\n' + '='*70)
print('  SPEARMAN CORRELATION: Pollutants vs HI and PET')
print('='*70)

for sid, sdata in STATION_DATA.items():
    df  = sdata['df']
    name = sdata['name']
    corr_rows = []
    for p in POLL_FEATURES:
        row = {'Pollutant': p}
        for tgt in TARGETS:
            valid = df[[p, tgt]].dropna()
            if len(valid) > 50:
                r, pval = spearmanr(valid[p], valid[tgt])
                row[f'r_{tgt}']    = round(r, 4)
                row[f'p_{tgt}']    = round(pval, 4)
                row[f'sig_{tgt}']  = '***' if pval<0.001 else ('**' if pval<0.01 else ('*' if pval<0.05 else ''))
                row[f'n_{tgt}']    = len(valid)
            else:
                row[f'r_{tgt}'] = np.nan; row[f'p_{tgt}'] = np.nan
                row[f'sig_{tgt}'] = ''; row[f'n_{tgt}'] = len(valid)
        corr_rows.append(row)

    corr_df = pd.DataFrame(corr_rows)
    CORR_RESULTS[sid] = corr_df
    print(f'\n  {name} ({sid}):')
    print(f'  {"Pollutant":<12} {"r_HI":>8} {"sig_HI":>6} {"r_PET":>8} {"sig_PET":>7}')
    print(f'  {"-"*48}')
    for _, row in corr_df.iterrows():
        print(f'  {row["Pollutant"]:<12} {row["r_HI"]:>8.4f} {row["sig_HI"]:>6} '
              f'{row["r_PET"]:>8.4f} {row["sig_PET"]:>7}')
    corr_df.to_csv(os.path.join(OUTPUT_DIR, f'{sid}_spearman_correlation.csv'), index=False)

print('\n✅ Correlation tables saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — PLOT 1: Side-by-Side Spearman Correlation Heatmap               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Spearman Correlation: Pollutants vs HI & PET\n'
             'Santacruz (43003) and Colaba (43057) | 2019–2025',
             fontsize=13, fontweight='bold', y=1.02)

for ax, (sid, sdata) in zip(axes, STATION_DATA.items()):
    corr_df = CORR_RESULTS[sid]
    heat = corr_df.set_index('Pollutant')[['r_HI','r_PET']].astype(float)
    heat.columns = ['HI (Heat Index)', 'PET']

    mask_sig = pd.DataFrame(False, index=heat.index, columns=heat.columns)
    for col, tgt in zip(heat.columns, TARGETS):
        mask_sig[col] = corr_df.set_index('Pollutant')[f'sig_{tgt}'].isin(['*','**','***'])

    sns.heatmap(heat, annot=True, fmt='.3f', cmap='RdBu_r',
                vmin=-0.5, vmax=0.5, center=0, ax=ax,
                linewidths=0.5, linecolor='white',
                annot_kws={'size': 10})
    # Add significance markers
    for i, poll in enumerate(heat.index):
        for j, tgt in enumerate(TARGETS):
            sig = corr_df.loc[corr_df['Pollutant']==poll, f'sig_{tgt}'].values[0]
            if sig:
                ax.text(j+0.85, i+0.25, sig, ha='center', va='center',
                        fontsize=8, color='black', fontweight='bold')

    ax.set_title(f'{sdata["name"]} ({sid})', fontsize=11, fontweight='bold')
    ax.set_xlabel('Thermal Stress Index', fontsize=10)
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'PLOT1_spearman_heatmap_both_stations.png'),
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Plot 1 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — PLOT 2: Station Comparison Bar Chart                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Spearman r — Station Comparison: Santacruz vs Colaba',
             fontsize=13, fontweight='bold')

colors_stn = {'43003': '#1D9E75', '43057': '#378ADD'}

for ax, tgt in zip(axes, TARGETS):
    x     = np.arange(len(POLL_FEATURES))
    width = 0.35
    for i, (sid, sdata) in enumerate(STATION_DATA.items()):
        vals  = CORR_RESULTS[sid].set_index('Pollutant')[f'r_{tgt}'].values.astype(float)
        bars  = ax.bar(x + (i-0.5)*width, vals, width,
                       label=f'{sdata["name"]} ({sid})',
                       color=colors_stn[sid], alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axhline( 0.1, color='gray', linewidth=0.5, linestyle=':')
    ax.axhline(-0.1, color='gray', linewidth=0.5, linestyle=':')
    ax.set_xticks(x)
    ax.set_xticklabels(POLL_FEATURES, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('Spearman r', fontsize=10)
    ax.set_ylim(-0.6, 0.6)
    ax.set_title(f'Correlation with {tgt}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.annotate('|r|>0.1 threshold', xy=(0, 0.12), fontsize=8, color='gray')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'PLOT2_station_comparison_bar.png'),
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Plot 2 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — PLOT 3: Monthly Correlation Heatmaps (Seasonal Pattern)         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Monthly Spearman Correlation: Pollutants vs HI & PET\n'
             'Santacruz (top) | Colaba (bottom)', fontsize=13, fontweight='bold')

for row_ax, (sid, sdata) in enumerate(STATION_DATA.items()):
    df   = sdata['df']
    for col_ax, tgt in enumerate(TARGETS):
        ax   = axes[row_ax][col_ax]
        heat = pd.DataFrame(index=POLL_FEATURES, columns=range(1,13), dtype=float)
        for m in range(1, 13):
            sub = df[df['MN']==m]
            for p in POLL_FEATURES:
                valid = sub[[p, tgt]].dropna()
                if len(valid) > 20:
                    r, _ = spearmanr(valid[p], valid[tgt])
                    heat.loc[p, m] = round(r, 3)
        heat.columns = MONTH_NAMES
        sns.heatmap(heat.astype(float), annot=True, fmt='.2f',
                    cmap='RdBu_r', vmin=-0.6, vmax=0.6, center=0,
                    ax=ax, linewidths=0.4, annot_kws={'size':7})
        ax.set_title(f'{sdata["name"]} ({sid}) — {tgt}', fontsize=10, fontweight='bold')
        ax.set_ylabel('Pollutant', fontsize=9)
        ax.set_xlabel('Month', fontsize=9)
        ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'PLOT3_monthly_correlation_heatmap.png'),
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Plot 3 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — PLOT 4: Box Plots — Pollutants by HI Category                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

KEY_POLLS = ['PM2.5', 'AQI_max', 'HAPSI', 'Ozone']

fig, axes = plt.subplots(len(KEY_POLLS), 2, figsize=(14, 4*len(KEY_POLLS)))
fig.suptitle('Pollutant Distribution by HI Heat Stress Category\n'
             'Santacruz (left) | Colaba (right)', fontsize=13, fontweight='bold')

colors_hi = ['#2ecc71','#f39c12','#e74c3c','#8e44ad']

for row_ax, poll in enumerate(KEY_POLLS):
    for col_ax, (sid, sdata) in enumerate(STATION_DATA.items()):
        ax  = axes[row_ax][col_ax]
        df  = sdata['df']
        if 'HI_Category' not in df.columns: continue
        groups = [df.loc[df['HI_Category']==c, poll].dropna().values
                  for c in sorted(df['HI_Category'].dropna().unique())]
        cat_labels = [HI_CATS.get(c, str(c))
                      for c in sorted(df['HI_Category'].dropna().unique())]
        bps = ax.boxplot(groups, patch_artist=True, showfliers=False,
                         medianprops=dict(color='black', linewidth=2))
        for patch, col in zip(bps['boxes'], colors_hi[:len(groups)]):
            patch.set_facecolor(col); patch.set_alpha(0.7)
        ax.set_xticklabels(cat_labels, rotation=20, ha='right', fontsize=8)
        ax.set_ylabel(f'{poll}', fontsize=9)
        ax.set_title(f'{sdata["name"]} — {poll} by HI Category', fontsize=9, fontweight='bold')
        # Add n count
        for i, g in enumerate(groups):
            ax.text(i+1, ax.get_ylim()[1]*0.95, f'n={len(g)}',
                    ha='center', fontsize=7, color='gray')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'PLOT4_boxplot_poll_by_HI_category.png'),
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Plot 4 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — PLOT 5: Box Plots — Pollutants by PET Category                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(len(KEY_POLLS), 2, figsize=(16, 4*len(KEY_POLLS)))
fig.suptitle('Pollutant Distribution by PET Category\n'
             'Santacruz (left) | Colaba (right)', fontsize=13, fontweight='bold')

colors_pet = ['#27ae60','#f1c40f','#e67e22','#e74c3c','#8e44ad']

for row_ax, poll in enumerate(KEY_POLLS):
    for col_ax, (sid, sdata) in enumerate(STATION_DATA.items()):
        ax  = axes[row_ax][col_ax]
        df  = sdata['df']
        if 'PET_Category' not in df.columns: continue
        cats = sorted(df['PET_Category'].dropna().unique())
        groups = [df.loc[df['PET_Category']==c, poll].dropna().values for c in cats]
        cat_labels = [PET_CATS.get(c, str(c)) for c in cats]
        bps = ax.boxplot(groups, patch_artist=True, showfliers=False,
                         medianprops=dict(color='black', linewidth=2))
        for patch, col in zip(bps['boxes'], colors_pet[:len(cats)]):
            patch.set_facecolor(col); patch.set_alpha(0.7)
        ax.set_xticklabels(cat_labels, rotation=25, ha='right', fontsize=8)
        ax.set_ylabel(poll, fontsize=9)
        ax.set_title(f'{sdata["name"]} — {poll} by PET Category', fontsize=9, fontweight='bold')
        for i, g in enumerate(groups):
            ax.text(i+1, ax.get_ylim()[1]*0.95, f'n={len(g)}',
                    ha='center', fontsize=7, color='gray')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'PLOT5_boxplot_poll_by_PET_category.png'),
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Plot 5 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — PLOT 6: Seasonal Correlation Radar Charts                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

SEASONS   = ['Winter', 'Pre-monsoon', 'Monsoon', 'Post-monsoon']
SEA_COLOR = {'Winter':'#3498db', 'Pre-monsoon':'#e67e22',
             'Monsoon':'#27ae60', 'Post-monsoon':'#9b59b6'}

fig, axes = plt.subplots(2, 2, figsize=(16, 12),
                          subplot_kw=dict(polar=True))
fig.suptitle('Seasonal Spearman r — Pollutants vs HI & PET\n'
             'Santacruz (top) | Colaba (bottom)', fontsize=13, fontweight='bold')

polls_radar = ['PM2.5','PM10','NO2','NH3','SO2','CO','Ozone','AQI_max']
N = len(polls_radar)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

for row_ax, (sid, sdata) in enumerate(STATION_DATA.items()):
    df = sdata['df']
    for col_ax, tgt in enumerate(TARGETS):
        ax = axes[row_ax][col_ax]
        for sea in SEASONS:
            sub = df[df['Season']==sea]
            vals = []
            for p in polls_radar:
                valid = sub[[p, tgt]].dropna()
                if len(valid) > 30:
                    r, _ = spearmanr(valid[p], valid[tgt])
                    vals.append(max(-1, min(1, r)))
                else:
                    vals.append(0)
            vals += vals[:1]
            ax.plot(angles, vals, 'o-', linewidth=1.5,
                    label=sea, color=SEA_COLOR[sea])
            ax.fill(angles, vals, alpha=0.08, color=SEA_COLOR[sea])

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(polls_radar, size=8)
        ax.set_ylim(-0.6, 0.6)
        ax.set_yticks([-0.4,-0.2,0,0.2,0.4])
        ax.set_yticklabels(['-0.4','-0.2','0','0.2','0.4'], size=7)
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.set_title(f'{sdata["name"]} ({sid}) — {tgt}',
                     size=10, fontweight='bold', pad=15)
        if row_ax==0 and col_ax==0:
            ax.legend(loc='upper right', bbox_to_anchor=(1.35,1.15), fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'PLOT6_seasonal_radar_charts.png'),
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Plot 6 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — PLOT 7: Scatter Plots — Top Correlated Pollutants               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Pick top 3 pollutants by |r| for each station × target
def top_polls(sid, tgt, n=3):
    corr = CORR_RESULTS[sid].copy()
    corr['abs_r'] = corr[f'r_{tgt}'].abs()
    return corr.nlargest(n, 'abs_r')['Pollutant'].tolist()

fig, axes = plt.subplots(4, 3, figsize=(15, 18))
fig.suptitle('Scatter Plots — Top Correlated Pollutants vs HI/PET\n'
             'Santacruz (rows 1–2) | Colaba (rows 3–4)', fontsize=12, fontweight='bold')

row = 0
for sid, sdata in STATION_DATA.items():
    df = sdata['df']
    for tgt in TARGETS:
        top3 = top_polls(sid, tgt)
        for col_ax, poll in enumerate(top3):
            ax   = axes[row][col_ax]
            valid = df[[poll, tgt]].dropna()
            r, pval = spearmanr(valid[poll], valid[tgt])
            # Hexbin for dense data
            hb = ax.hexbin(valid[poll], valid[tgt],
                           gridsize=30, cmap='YlOrRd', mincnt=1)
            plt.colorbar(hb, ax=ax, shrink=0.7)
            # OLS trend line
            m_fit, b_fit = np.polyfit(valid[poll], valid[tgt], 1)
            x_line = np.linspace(valid[poll].min(), valid[poll].max(), 100)
            ax.plot(x_line, m_fit*x_line+b_fit, 'b--', linewidth=1.5, alpha=0.8)
            ax.set_xlabel(poll, fontsize=9)
            ax.set_ylabel(tgt, fontsize=9)
            ax.set_title(f'{sdata["name"]} | {poll} vs {tgt}\n'
                         f'r={r:.3f}  p={pval:.1e}  n={len(valid):,}',
                         fontsize=8, fontweight='bold')
        row += 1

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'PLOT7_scatter_top_pollutants.png'),
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Plot 7 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — PLOT 8: Monthly Mean Time Series — Pollution + HI/PET          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(4, 1, figsize=(16, 16))
fig.suptitle('Monthly Mean Time Series: Pollution & Thermal Stress (2019–2025)',
             fontsize=13, fontweight='bold')

panel_config = [
    ('PM2.5',   '#e74c3c',  'PM2.5 (μg/m³)'),
    ('AQI_max', '#e67e22',  'AQI_max (NAQI)'),
    ('HI',      '#c0392b',  'Heat Index (°C)'),
    ('PET',     '#8e44ad',  'PET (°C)'),
]

for ax, (col, color, ylabel) in zip(axes, panel_config):
    for sid, sdata in STATION_DATA.items():
        df    = sdata['df']
        if col not in df.columns: continue
        df_t  = df.copy()
        df_t['YM'] = df_t['YEAR'].astype(str) + '-' + df_t['MN'].astype(str).str.zfill(2)
        monthly = df_t.groupby('YM')[col].mean().reset_index()
        monthly['date'] = pd.to_datetime(monthly['YM'], format='%Y-%m')
        monthly = monthly.sort_values('date')
        ls = '-' if sid == '43003' else '--'
        lc = '#1D9E75' if sid == '43003' else '#378ADD'
        ax.plot(monthly['date'], monthly[col], ls, color=lc, linewidth=1.5,
                label=f'{sdata["name"]} ({sid})', alpha=0.9)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    # Shade monsoon months (Jun–Sep)
    for year in range(2019, 2026):
        ax.axvspan(pd.Timestamp(f'{year}-06-01'), pd.Timestamp(f'{year}-09-30'),
                   alpha=0.08, color='blue')

axes[-1].set_xlabel('Date', fontsize=10)
axes[0].annotate('Blue shading = Monsoon (Jun–Sep)', xy=(0.01, 0.92),
                 xycoords='axes fraction', fontsize=8, color='blue')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'PLOT8_monthly_timeseries.png'),
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Plot 8 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 14 — PLOT 9: AQI Category × HI Category Cross-Tab Heatmap          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def aqi_label(x):
    if pd.isna(x): return np.nan
    if x<=50:  return 'Good'
    if x<=100: return 'Satisfactory'
    if x<=200: return 'Moderate'
    if x<=300: return 'Poor'
    if x<=400: return 'Very Poor'
    return 'Severe'

AQI_ORDER = ['Good','Satisfactory','Moderate','Poor','Very Poor','Severe']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Cross-Tabulation: AQI Category × HI/PET Category (%)\n'
             'Santacruz (top) | Colaba (bottom)', fontsize=13, fontweight='bold')

for row_ax, (sid, sdata) in enumerate(STATION_DATA.items()):
    df = sdata['df'].copy()
    df['AQI_Cat'] = df['AQI_max'].apply(aqi_label)

    for col_ax, (tgt, cat_dict) in enumerate(zip(TARGETS, [HI_CATS, PET_CATS])):
        ax   = axes[row_ax][col_ax]
        tgt_col = f'{tgt}_Category'
        if tgt_col not in df.columns: continue
        df_ct = df[[tgt_col,'AQI_Cat']].dropna()
        df_ct[tgt_col] = df_ct[tgt_col].map(cat_dict)
        ct = pd.crosstab(df_ct['AQI_Cat'], df_ct[tgt_col], normalize='columns') * 100
        ct = ct.reindex([x for x in AQI_ORDER if x in ct.index])

        sns.heatmap(ct, annot=True, fmt='.1f', cmap='YlOrRd',
                    vmin=0, vmax=70, ax=ax,
                    linewidths=0.5, annot_kws={'size':8})
        ax.set_title(f'{sdata["name"]} — AQI vs {tgt} (%)', fontsize=10, fontweight='bold')
        ax.set_xlabel(f'{tgt} Category', fontsize=9)
        ax.set_ylabel('AQI Category', fontsize=9)
        ax.tick_params(axis='x', rotation=25, labelsize=8)
        ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'PLOT9_AQI_vs_HI_PET_crosstab.png'),
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Plot 9 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 15 — PLOT 10: Diurnal Pattern — Pollution vs HI/PET (hourly mean)   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Diurnal Pattern: Pollution & Thermal Stress (Hourly Mean)\n'
             'Santacruz (top) | Colaba (bottom)', fontsize=13, fontweight='bold')

diurnal_polls = ['PM2.5','Ozone','AQI_max']
colors_d = ['#e74c3c','#e67e22','#8e44ad']

for row_ax, (sid, sdata) in enumerate(STATION_DATA.items()):
    df = sdata['df'].copy()
    df['IST_Hour'] = df['_dt'].dt.hour if '_dt' in df.columns else df['MN'].map({})

    # Reconstruct hour from MN (month) won't work — use HR_IST_MAP
    # Use the synoptic hours from HR_IST_MAP
    df['IST_Hour'] = df['HR'].map(HR_IST_MAP)
    df_valid = df.dropna(subset=['IST_Hour'])

    ax1 = axes[row_ax][0]
    ax2 = axes[row_ax][1]

    # Left: Pollution diurnal
    for poll, col in zip(diurnal_polls, colors_d):
        if poll not in df_valid.columns: continue
        hourly = df_valid.groupby('IST_Hour')[poll].mean()
        ax1.plot(hourly.index, hourly.values, 'o-', color=col,
                 linewidth=2, markersize=5, label=poll)
    ax1.set_xlabel('Hour (IST)', fontsize=10)
    ax1.set_ylabel('Concentration (μg/m³ or NAQI)', fontsize=9)
    ax1.set_title(f'{sdata["name"]} — Diurnal Pollution Pattern', fontsize=10, fontweight='bold')
    ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
    ax1.set_xticks(sorted(df_valid['IST_Hour'].unique()))

    # Right: HI + PET diurnal
    ax2_twin = ax2.twinx()
    for tgt, col, ax_use in zip(TARGETS, ['#c0392b','#8e44ad'], [ax2, ax2_twin]):
        hourly_t = df_valid.groupby('IST_Hour')[tgt].mean()
        ax_use.plot(hourly_t.index, hourly_t.values, 'o--', color=col,
                    linewidth=2, markersize=5, label=tgt)
        ax_use.set_ylabel(f'{tgt} (°C)', fontsize=9, color=col)
        ax_use.tick_params(axis='y', colors=col)
    ax2.set_xlabel('Hour (IST)', fontsize=10)
    ax2.set_title(f'{sdata["name"]} — Diurnal HI & PET Pattern', fontsize=10, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.set_xticks(sorted(df_valid['IST_Hour'].unique()))
    lines1, labs1 = ax2.get_legend_handles_labels()
    lines2, labs2 = ax2_twin.get_legend_handles_labels()
    ax2.legend(lines1+lines2, labs1+labs2, fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'PLOT10_diurnal_patterns.png'),
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Plot 10 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 16 — PLOT 11: HAPSI Distribution by Season and Station              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('HAPSI Distribution by Season — Santacruz vs Colaba',
             fontsize=13, fontweight='bold')

sea_order  = ['Winter','Pre-monsoon','Monsoon','Post-monsoon']
sea_colors = ['#3498db','#e67e22','#27ae60','#9b59b6']

for ax, (sid, sdata) in zip(axes, STATION_DATA.items()):
    df = sdata['df']
    data_by_sea = [df.loc[df['Season']==s,'HAPSI'].dropna().values for s in sea_order]
    vp = ax.violinplot(data_by_sea, positions=range(len(sea_order)),
                       showmedians=True, showextrema=True)
    for body, col in zip(vp['bodies'], sea_colors):
        body.set_facecolor(col); body.set_alpha(0.6)
    vp['cmedians'].set_color('black'); vp['cmedians'].set_linewidth(2)
    ax.set_xticks(range(len(sea_order)))
    ax.set_xticklabels(sea_order, rotation=15, ha='right')
    ax.set_ylabel('HAPSI (0–100)', fontsize=10)
    ax.set_title(f'{sdata["name"]} ({sid})\n'
                 f'α={sdata["alpha"]:.2f}  β={sdata["beta"]:.2f}  γ={sdata["gamma"]:.2f}',
                 fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    for i, (s, d) in enumerate(zip(sea_order, data_by_sea)):
        ax.text(i, ax.get_ylim()[1]*0.97, f'n={len(d):,}',
                ha='center', va='top', fontsize=7, color='gray')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'PLOT11_HAPSI_violin_by_season.png'),
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Plot 11 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 17 — PLOT 12: Pollutant-Pollutant Correlation Matrix (Both Stations)║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Pollutant–Pollutant Spearman Correlation Matrix\n'
             'Santacruz (left) | Colaba (right)', fontsize=13, fontweight='bold')

for ax, (sid, sdata) in zip(axes, STATION_DATA.items()):
    df     = sdata['df']
    p_cols = [p for p in POLL_FEATURES if p in df.columns]
    corr_m = df[p_cols].corr(method='spearman')
    mask   = np.triu(np.ones_like(corr_m, dtype=bool), k=1)
    sns.heatmap(corr_m, annot=True, fmt='.2f', cmap='RdBu_r',
                vmin=-1, vmax=1, center=0, ax=ax,
                mask=mask, linewidths=0.4,
                annot_kws={'size': 7})
    ax.set_title(f'{sdata["name"]} ({sid})', fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=40, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'PLOT12_pollutant_corr_matrix.png'),
            dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('✅ Plot 12 saved.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 18 — Final Summary Table                                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print('\n' + '='*70)
print('  FINAL SUMMARY — Pollution × Heat Stress Correlation')
print('  Santacruz (43003) and Colaba (43057) | 2019–2025')
print('='*70)

for sid, sdata in STATION_DATA.items():
    print(f'\n  {sdata["name"]} ({sid}):')
    corr = CORR_RESULTS[sid]

    for tgt in TARGETS:
        sig_polls = corr[corr[f'sig_{tgt}'].isin(['*','**','***'])]['Pollutant'].tolist()
        strong    = corr[corr[f'r_{tgt}'].abs() >= 0.2]['Pollutant'].tolist()
        top1      = corr.loc[corr[f'r_{tgt}'].abs().idxmax(), 'Pollutant']
        top1_r    = corr.loc[corr[f'r_{tgt}'].abs().idxmax(), f'r_{tgt}']
        print(f'    {tgt}: top correlated = {top1} (r={top1_r:.3f})')
        print(f'         significant pollutants: {sig_polls}')
        print(f'         |r| ≥ 0.2: {strong}')

print('\n  Plots saved:')
plots = ['PLOT1_spearman_heatmap_both_stations',
         'PLOT2_station_comparison_bar',
         'PLOT3_monthly_correlation_heatmap',
         'PLOT4_boxplot_poll_by_HI_category',
         'PLOT5_boxplot_poll_by_PET_category',
         'PLOT6_seasonal_radar_charts',
         'PLOT7_scatter_top_pollutants',
         'PLOT8_monthly_timeseries',
         'PLOT9_AQI_vs_HI_PET_crosstab',
         'PLOT10_diurnal_patterns',
         'PLOT11_HAPSI_violin_by_season',
         'PLOT12_pollutant_corr_matrix']
for p in plots:
    print(f'    {p}.png')

print(f'\n  Output directory: {OUTPUT_DIR}')
print('='*70)
print('✅ ALL DONE.')